# Test Stage 2 — Is the acquisition system alive, and is its clock right?

**What this stage checks:** that the new hardware responds to software at all, that its
internal timing is correct, and that the reference tone it generates is at the frequency
you asked for.

Nothing here depends on the mixer or the decimator being *correct* — those come in stages
3 and 4. This stage answers the duller question that has to be answered first: **is the
module there, and is it running at the right speed?**

---

## What you need

| | |
|---|---|
| Bitstream | The one containing `red_pitaya_acq` — confirm with `which_bitstream.py` |
| Jupyter | Started as **root** (`sudo jupyter notebook`), because this reads `/dev/mem` |
| Signal generator | Only for Check 4. A **low-frequency** sine, around 1–10 kHz, a few hundred mV, into **IN1** |

## The four checks

1. **Registers respond** — can we write a value and read the same value back?
2. **The chain runs at the right rate** — does it produce exactly 125 MHz ÷ R samples per second?
3. **Reset works** — does the sample counter go back to zero?
4. **The reference tone is at the right frequency** — measured directly.

## Why a low-frequency input for Check 4

The reference tone is generated *inside* the chip and has no pin of its own, so it cannot
be probed directly. But the mixer multiplies the input by it — so if you feed in something
that barely changes (a slow, low-frequency signal is near-constant over one cycle of a
40 MHz tone), the mixer output becomes **a copy of the reference tone itself**, slowly
amplitude-modulated.

Measuring the frequency of that copy measures the reference tone.

---

### Simulation or the real trap?

**This stage never touches the emulated coil.** It measures the acquisition chain itself —
the part that listens — using a bench signal generator plugged into IN1. That is already a
real hardware measurement, so nothing here changes when the trap is connected.

One thing *is* worth doing again for real work. These tests run at a convenient
`F_SIG = 10 MHz` and one fixed averaging setting, and droop, noise and timing all depend on
both. **Before taking trap data, re-run this stage at the frequency and averaging you will
actually use** — around 40 MHz for the coil — so the instrument is characterised where it
is used rather than where it was convenient to test.


## Setup — map the hardware

Run this cell first. It defines the driver used by every cell below.

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
# =============================================================================
#  Full register map is in the header of red_pitaya_acq.v and in
#  docs/acquisition.md.
#
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer that the GP-AXI to system-bus bridge cannot
#  acknowledge; it locks the bus and reboots the board. Do not "optimise" the
#  loops below into slice operations.
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0] enable  [1] soft reset  [2] test mode  [3] scope tap
R_LO_FINC    = 0x04   # local-oscillator frequency word
R_DEC_LOG2   = 0x08   # log2 of the decimation factor, 6..12
R_SRC_SEL    = 0x0C   # 0 = IN1, 1 = IN2
R_TAP_SEL    = 0x10   # 0 = mixer output, 1 = CIC output
R_TAP_SHIFT  = 0x14   # which 14-bit window of the tapped value to show
R_PEEK_I     = 0x18   # most recent I sample (read-only)
R_PEEK_Q     = 0x1C   # most recent Q sample (read-only)
R_SAMPLE_CNT = 0x20   # output samples since reset (read-only)
R_STATUS     = 0x24   # [0] enabled [1] test mode [2] tap active (read-only)


def s32(v):
    """Reinterpret a 32-bit register value as a signed number.

    Registers come back as unsigned integers, but I and Q are signed. Anything
    with the top bit set is negative, and the value is 2**32 lower than it looks.
    """
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    # ---- frequency word -----------------------------------------------------
    @staticmethod
    def finc(f_lo):
        """Convert a wanted local-oscillator frequency [Hz] into the register value.

        The oscillator is a counter that represents an angle; its full range is
        one complete turn. Adding `finc` every clock tick advances the angle, so

            f_out = f_clk * finc / 2**32     =>     finc = f_out / f_clk * 2**32

        Step size is 125e6 / 2**32 = 0.0291 Hz.
        """
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    @staticmethod
    def finc_to_hz(finc):
        """Inverse of finc(): the frequency the hardware will actually produce."""
        return finc * FS_CLK / (1 << 32)

    # ---- configuration ------------------------------------------------------
    def setup(self, f_lo, dec_log2, src=0, test=False,
              tap_en=False, tap_sel=0, tap_shift=10):
        """Configure and start the chain.

        THE ORDER MATTERS. The decimator holds a running total that assumes one
        fixed decimation factor; changing it while data is flowing leaves a
        meaningless partial sum in the pipeline and produces a burst of nonsense.
        So the sequence is always: disable, write settings, reset, enable.
        """
        self.w(R_CTRL, 0)                       # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))    # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        ctrl = 1 | (4 if test else 0) | (8 if tap_en else 0)
        self.w(R_CTRL, ctrl | 2)                # 3. enable + one-shot reset
        self.w(R_CTRL, ctrl)                    # 4. running

    def disable(self):
        self.w(R_CTRL, 0)

    def reset(self):
        """Pulse the soft reset, leaving the other control bits as they are."""
        ctrl = self.r(R_CTRL) & 0xD             # keep enable / test / tap bits
        self.w(R_CTRL, ctrl | 2)
        self.w(R_CTRL, ctrl)

    # ---- readback -----------------------------------------------------------
    def peek(self):
        """Most recent (I, Q) output sample, as signed integers."""
        return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))

    def count(self):
        return self.r(R_SAMPLE_CNT)


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
print('STATUS = 0x%08X' % acq.r(R_STATUS))

In [ ]:
# =============================================================================
#  Internal scope driver  -  register block at 0x40100000 (sys[1])
# =============================================================================
#  Taken unchanged from notebooks/feedback/scope_phase_check.ipynb, which is
#  already proven on this board.
#
#  NOTE: the equalisation filter this once configured has been REMOVED from
#  the design (it caused 1173 timing violations and was switched off anyway -
#  see CHANGELOG.md). The coefficient registers still exist and still accept
#  writes, they simply have no effect. For these measurements that is an
#  improvement: the scope is now a flat, unmodified path.
# =============================================================================
SC_BASE = 0x40100000
SC_MAP  = 0x40000
BUF_A   = 0x10000
BUF_B   = 0x20000
NMAX    = 1 << 14        # 16384 samples per channel


class Scope(object):
    def __init__(self):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, SC_MAP, offset=SC_BASE)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    def _buf(self, off, n):
        # One 32-bit word at a time - see the warning in the acquisition driver.
        m  = self.m
        ba = bytearray(4 * n)
        for i in range(n):
            o = off + 4 * i
            ba[4*i:4*i+4] = m[o:o+4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        # The buffer holds 14-bit two's-complement values. Anything with bit 13
        # set is negative and must have 0x4000 subtracted.
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=NMAX):
        self.w(0x14, dec)       # scope's own decimation (1 = every sample)
        self.w(0x10, n - 1)     # how many samples to record
        self.w(0x00, 2)         # reset the acquisition engine
        self.w(0x00, 1)         # arm
        self.w(0x04, 1)         # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:            # clears itself when the capture ends
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        return self._buf(BUF_A, n), self._buf(BUF_B, n), FS_CLK / dec


sc = Scope()
print('internal scope mapped at 0x%08X' % SC_BASE)

---
## Check 1 — Do the registers respond?

The simplest possible question: if we write a number into a register and read it back, do we
get the same number?

This sounds trivial but catches a surprising amount: a wrong base address, the module not
being included in the build, the bus slot still being stubbed out, or bits being dropped
because a register was declared too narrow.

**Passes if:** every line says `OK`.

**If it fails:** the module is not at `0x40600000`. Check the build actually included
`red_pitaya_acq.v`, and that the bus stub loop in `red_pitaya_top.sv` covers only `sys[7]`.

In [ ]:
# Each entry: (register offset, name, a test value, how many bits the register keeps)
tests = [
    (R_LO_FINC,   'LO_FINC',   0x12345678, 32),
    (R_LO_FINC,   'LO_FINC',   Acq.finc(40e6), 32),
    (R_DEC_LOG2,  'DEC_LOG2',  10,          4),
    (R_DEC_LOG2,  'DEC_LOG2',  6,           4),
    (R_SRC_SEL,   'SRC_SEL',   1,           1),
    (R_TAP_SEL,   'TAP_SEL',   1,           2),
    (R_TAP_SHIFT, 'TAP_SHIFT', 9,           5),
]

acq.disable()          # make sure nothing is running while we poke at registers
all_ok = True

print('%-10s %-12s %-12s %s' % ('register', 'written', 'read back', 'result'))
print('-' * 48)
for off, name, val, nbits in tests:
    acq.w(off, val)
    got  = acq.r(off)
    # Only compare the bits the register actually keeps - the rest read as zero,
    # which is correct behaviour, not a fault.
    want = val & ((1 << nbits) - 1)
    ok   = (got == want)
    all_ok &= ok
    print('%-10s 0x%08X   0x%08X   %s' % (name, val, got, 'OK' if ok else 'MISMATCH'))

print()
print('CHECK 1:', 'PASS' if all_ok else 'FAIL')

# Leave the registers in a sensible state for the checks that follow.
acq.w(R_SRC_SEL, 0)
acq.w(R_TAP_SEL, 0)

---
## Check 2 — Does the chain run at the right rate?

The decimator is supposed to produce one output sample for every **R** input samples, where
R is the decimation factor. With the board sampling at 125 million times a second, that means
the output rate should be exactly

$$\text{output rate} = \frac{125\ \text{MHz}}{R}$$

We measure it by reading the sample counter twice, a known time apart, and dividing.

**This check needs no signal generator.** The chain counts samples whether or not anything
useful is going into it, so this is purely a test of the internal timing.

**Passes if:** every measured rate is within about 1% of the prediction. The error comes from
Python's timing, not the hardware — the hardware's rate is exact by construction.

**If it fails:**
- All rates a constant factor out → the decimation counter is counting wrongly.
- Rate is zero → the chain is not running. Check the enable bit and `STATUS`.
- Only large R wrong → the counter is too narrow for the biggest factor.

In [ ]:
def measure_rate(dec_log2, seconds=0.30):
    """Count output samples over a known interval and return samples per second."""
    R = 1 << dec_log2
    acq.setup(f_lo=1e6, dec_log2=dec_log2)   # LO frequency is irrelevant here
    time.sleep(0.02)                          # let it settle

    c0 = acq.count(); t0 = time.time()
    time.sleep(seconds)
    c1 = acq.count(); t1 = time.time()

    # The counter is 32 bits and wraps. At the fastest rate (R=64, ~1.95 MHz) it
    # takes ~37 minutes to wrap, so over 0.3 s a wrap is impossible - but the
    # masking below costs nothing and makes the intent explicit.
    delta = (c1 - c0) & 0xFFFFFFFF
    return R, delta / (t1 - t0), delta


print('%6s %14s %14s %10s  %s' % ('R', 'expected [Hz]', 'measured [Hz]', 'error', 'result'))
print('-' * 62)
all_ok = True
for L in range(6, 13):
    R, rate, n = measure_rate(L)
    expect = FS_CLK / R
    err    = (rate - expect) / expect
    ok     = abs(err) < 0.02 and n > 0
    all_ok &= ok
    print('%6d %14.1f %14.1f %9.2f%%  %s'
          % (R, expect, rate, 100 * err, 'OK' if ok else 'FAIL'))

print()
print('CHECK 2:', 'PASS' if all_ok else 'FAIL')

---
## Check 3 — Does the reset work?

The decimator accumulates a running total that assumes one fixed decimation factor. If you
change the factor without resetting, that partial sum is left behind and the next few output
samples are meaningless.

So the software always follows the sequence **disable → change settings → reset → enable**,
and this check confirms the reset half of that actually does something.

**Passes if:** the counter is large before the reset and small immediately after.

**If it fails:** the soft-reset bit is not reaching the logic, and every later measurement
will carry contamination from whatever the chain was doing beforehand.

In [ ]:
acq.setup(f_lo=1e6, dec_log2=8)      # R = 256, output rate ~488 kHz
time.sleep(0.20)

before = acq.count()
acq.reset()
time.sleep(0.001)                     # ~500 samples' worth at this rate
after  = acq.count()

print('samples counted before reset : %d' % before)
print('samples counted after  reset : %d' % after)
print()

ok = (before > 1000) and (after < before / 10)
print('CHECK 3:', 'PASS' if ok else 'FAIL')
if not ok and before <= 1000:
    print('   the chain was barely running - check Check 2 passed first')

---
## Check 4 — Is the reference tone at the frequency we asked for?

Now the real question of this stage.

**Wiring:** signal generator → **IN1**. Set it to a **low frequency, 1–10 kHz**, sine, a few
hundred millivolts. Nothing else needs connecting.

**Why this works.** Over one cycle of a megahertz-scale reference tone, a 5 kHz input is
essentially a constant. Multiplying a constant by the reference tone gives back the reference
tone, scaled. So the mixer output is a copy of the internal oscillator, and we can measure its
frequency with the scope.

We look at it through the **scope tap**, which substitutes the mixer output for the scope's
normal input — so the scope and the notebook you already have do the capturing.

Set `F_LO_TEST` below to something the scope can measure comfortably. A few hundred kHz to a
few MHz is ideal: fast enough to be clearly above the input frequency, slow enough that a
16384-sample capture contains many cycles.

**Passes if:** the measured frequency matches the requested one to better than 0.1%.

**If it fails:**
- Measured frequency is a constant multiple of the request → the frequency-word formula is wrong.
- Nothing but noise → the mixer is not receiving the input. Check the generator is on IN1 and `SRC_SEL` is 0.
- Amplitude near zero → increase the generator amplitude, or reduce `TAP_SHIFT`.

In [ ]:
F_LO_TEST = 1.0e6          # <-- reference-tone frequency to test [Hz]
F_IN_HINT = 5e3            # <-- roughly what your generator is set to [Hz], for reference


def autorange_tap(tap_sel, start=12):
    """Pick the TAP_SHIFT that uses the scope's range without clipping.

    The tapped value is wider than the scope's 14-bit input, so only a 14-bit
    window of it is sent. TAP_SHIFT chooses where that window sits. Too high and
    the signal is a few counts; too low and it clips at +-8191.

    This walks the window down until the signal is comfortably large, then stops
    one step before it would clip.
    """
    best = start
    for shift in range(start, -1, -1):
        acq.w(R_TAP_SHIFT, shift)
        time.sleep(0.01)
        a, b, _ = sc.acquire(dec=1, n=4096)
        peak = max(np.abs(a).max(), np.abs(b).max())
        if peak > 7500:            # about to clip - back off one step
            break
        best = shift
        if peak > 2000:            # plenty of range, no need to go further
            break
    acq.w(R_TAP_SHIFT, best)
    return best


# Configure: reference tone at F_LO_TEST, scope tap showing the MIXER output.
acq.setup(f_lo=F_LO_TEST, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=12)
time.sleep(0.05)

shift = autorange_tap(tap_sel=0)
a, b, fs = sc.acquire(dec=1)

# Estimate the dominant frequency, with parabolic interpolation between FFT bins
# so the answer is not limited to the bin spacing of 125e6/16384 = 7.6 kHz.
def est_freq(x, fs):
    x = x - x.mean()
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    k = int(np.argmax(X[1:])) + 1
    if 1 <= k < len(X) - 1:                      # refine using the two neighbours
        l, c, r = (np.log(X[k-1] + 1e-9), np.log(X[k] + 1e-9), np.log(X[k+1] + 1e-9))
        k = k + 0.5 * (l - r) / (l - 2*c + r)
    return k * fs / len(x)

f_meas   = est_freq(a, fs)
f_actual = Acq.finc_to_hz(Acq.finc(F_LO_TEST))   # what the hardware can exactly produce
err      = (f_meas - f_actual) / f_actual

print('TAP_SHIFT chosen        : %d' % shift)
print('signal amplitude on tap : %.0f counts (I),  %.0f counts (Q)'
      % (np.abs(a).max(), np.abs(b).max()))
print()
print('requested reference tone: %12.3f Hz' % F_LO_TEST)
print('hardware can produce    : %12.3f Hz   (frequency word %d)'
      % (f_actual, Acq.finc(F_LO_TEST)))
print('measured on the tap     : %12.3f Hz' % f_meas)
print('error                   : %+.4f %%' % (100 * err))
print()

ok = abs(err) < 1e-3
print('CHECK 4:', 'PASS' if ok else 'FAIL')
if not ok:
    print('   if the measurement is close to a simple multiple of the request,')
    print('   suspect the frequency-word formula rather than the hardware')

### Look at it

The two traces are the in-phase and quadrature copies of the reference tone. They should
be clean sine waves **a quarter of a cycle apart** — that 90° relationship is what makes the
mixer able to tell frequencies above the reference from frequencies below it.

In [ ]:
import matplotlib.pyplot as plt

per = fs / f_meas
win = int(min(len(a), round(4 * per)))       # about four cycles
t   = np.arange(win) / fs * 1e6              # microseconds

plt.figure(figsize=(9, 4))
plt.plot(t, a[:win], label='I  (input x cos)')
plt.plot(t, b[:win], label='Q  (input x -sin)')
plt.xlabel('time [us]')
plt.ylabel('tapped value [scope counts]')
plt.title('Reference tone seen through the mixer  (%.3f kHz)' % (f_meas / 1e3))
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# The two should be 90 degrees apart. Measure it, as a bonus check on the
# oscillator's quarter-turn table lookup.
n  = np.arange(len(a))
c  = np.cos(2 * np.pi * f_meas * n / fs)
s  = np.sin(2 * np.pi * f_meas * n / fs)
pa = np.dot(a - a.mean(), c) - 1j * np.dot(a - a.mean(), s)
pb = np.dot(b - b.mean(), c) - 1j * np.dot(b - b.mean(), s)
dphi = np.degrees(np.angle(pb) - np.angle(pa))
dphi = (dphi + 180) % 360 - 180

print('phase(Q) - phase(I) = %+.2f deg    (expect +-90 deg)' % dphi)
print('quadrature error    = %+.2f deg' % (abs(dphi) - 90))

---
## Stage 2 summary

| Check | What it proves |
|---|---|
| 1 — registers | The module is present, at the right address, and software can talk to it |
| 2 — sample rate | The decimation counter is right and the chain is running |
| 3 — reset | Settings can be changed cleanly, without contamination from before |
| 4 — reference tone | The oscillator produces the frequency you ask for |

**All four passing** means the foundations are sound: the hardware is there, it runs at the
right speed, and its internal reference is accurate. None of it says the mixer or the
decimator are *correct* — that is stages 3 and 4.

**Before moving on**, switch the scope tap off so the scope behaves normally again:

In [ ]:
acq.disable()
print('acquisition disabled, scope restored to its normal inputs')
print('STATUS = 0x%08X' % acq.r(R_STATUS))